Multivariate imputation by chained Equation For missing values
                        (MICE)

types of missing categories data
1->missing completely at random (MCAR) --mila hi nahi
2->missing at random(MAR)  ---kuch kuch mila nahi mode se bhar sakate
3->Missing not at random
                                    When to use (MICE)  ??
Advantage:
            Accurate
        
Disadvantage:
            Slow

In [63]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [64]:
df=np.round(pd.read_csv("40_Startups.csv")[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df=df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [65]:
df=df.iloc[:,0:-1]
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [66]:
df.iloc[1,0]=np.NaN
df.iloc[3,1]=np.NaN
df.iloc[-1,-1]=np.NaN
df.head()

C:\Users\niraj\AppData\Local\Temp\ipykernel_168880\4256394514.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1,0]=np.NaN
C:\Users\niraj\AppData\Local\Temp\ipykernel_168880\4256394514.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3,1]=np.NaN
C:\Users\niraj\AppData\Local\Temp\ipykernel_168880\4256394514.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1,-1]=np.NaN


,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


In [67]:
#step 1 -impute all missing values with mean of respective col

df0=pd.DataFrame()
df0['R&D Spend']=df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration']=df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend']=df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [68]:
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [69]:
#remove the coll imputed value
df1=df0.copy()
df1.iloc[1,0]=np.NaN
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


In [70]:
#use first 3 rows to build a model and use the last for predication
x=df1.iloc[[0,2,3,4],1:3]
x

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


In [71]:
y=df1.iloc[[0,2,3,4],0]
y

21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

In [72]:
lr=LinearRegression()
lr.fit(x,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

c:\Users\niraj\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [73]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # This enables IterativeImputer
from sklearn.impute import IterativeImputer

# Sample data (as you have described)
data = {
    'R&D Spend': [8.0, 4.0, 15.0, 12.0, 2.0],
    'Administration': [15.0, 5.0, 10.0, 16.0, 15.0],
    'Marketing Spend': [30.0, 20.0, 41.0, 26.0, 3.0],
    'Profit': [11.0, 9.0, 19.0, 13.0, 7.0]
}

# Create a DataFrame
df = pd.DataFrame(data)

# Introduce NaN values randomly
np.random.seed(9)  # Ensuring reproducibility
missing_rate = 0.2  # Set missing rate (e.g., 20% of the values will be missing)
n_missing = int(np.floor(missing_rate * df.size))  # Number of values to be replaced with NaN
missing_indices = np.random.choice(df.size, size=n_missing, replace=False)  # Randomly select indices

# Replace the selected indices with NaN
df.values.ravel()[missing_indices] = np.nan

print("Data with missing values:")
print(df)

# 1. Apply Iterative Imputation (Automatic Method)

# Initialize IterativeImputer
iterative_imputer = IterativeImputer(random_state=0, max_iter=10)  # You can control iterations

# Fit the imputer and transform the data
imputed_df = iterative_imputer.fit_transform(df)

# Convert the imputed array back to a dataframe for easy interpretation
imputed_df = pd.DataFrame(imputed_df, columns=df.columns)

print("\nData after Iterative Imputation (Automatic):")
print(np.round(imputed_df, 2))  # Round to 2 decimal places for easier viewing

# 2. Apply Iterative Imputation in a Loop (Custom Iteration Function)

def iterative_imputation_loop(df, max_iter=10):
    imputed_df = df.copy()  # Start with the original dataframe
    for i in range(max_iter):
        print(f"Iteration {i+1}:")
        iterative_imputer = IterativeImputer(random_state=0, max_iter=1)  # One iteration per loop
        imputed_df = iterative_imputer.fit_transform(imputed_df)  # Impute
        imputed_df = pd.DataFrame(imputed_df, columns=df.columns)  # Convert to DataFrame
        print(np.round(imputed_df, 2))  # Display after each iteration
    return imputed_df

# Apply the loop function
print("\nData after Iterative Imputation (Manual Loop):")
imputed_df_loop = iterative_imputation_loop(df)



Data with missing values:
   R&D Spend  Administration  Marketing Spend  Profit
0        8.0            15.0             30.0    11.0
1        4.0             5.0             20.0     9.0
2       15.0            10.0             41.0    19.0
3       12.0            16.0             26.0    13.0
4        2.0            15.0              3.0     7.0

Data after Iterative Imputation (Automatic):
   R&D Spend  Administration  Marketing Spend  Profit
0        8.0            15.0             30.0    11.0
1        4.0             5.0             20.0     9.0
2       15.0            10.0             41.0    19.0
3       12.0            16.0             26.0    13.0
4        2.0            15.0              3.0     7.0

Data after Iterative Imputation (Manual Loop):
Iteration 1:
   R&D Spend  Administration  Marketing Spend  Profit
0        8.0            15.0             30.0    11.0
1        4.0             5.0             20.0     9.0
2       15.0            10.0             41.0    19.0
3  